# Notebook 4 · Building a model: neural networks and Stable-Baselines3

**Time:** about 1½ hours.

**You'll learn**
- why a neural network replaces the Q-table
- how to build a small network in PyTorch, and what its inputs and outputs are
- how DQN extends the Q-learning you wrote in notebook 3
- how to train, monitor and evaluate an agent with Stable-Baselines3 and TensorBoard
- how to swap one algorithm for another, and why results differ
- how all of this maps onto the main Stratus project

> 📗 **This is the solutions copy.** The exercise version is `onboarding/04_neural_networks_and_sb3.ipynb`. Try the exercises there first.

## Setup

In [ ]:
import sys
from pathlib import Path

# toy_env.py lives in onboarding/. This makes it importable from both the
# exercise copy (onboarding/) and the solutions copy (onboarding/solutions/).
here = Path.cwd()
if (here / "onboarding" / "toy_env.py").is_file():
    here = here / "onboarding"
sys.path.insert(0, str(here.parent if here.name == "solutions" else here))

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from matplotlib.colors import ListedColormap
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.monitor import Monitor, load_results

from toy_env import (
    ACTION_NAMES,
    ALTITUDE_SCALE,
    BAND,
    EVAL_SEEDS,
    STAY,
    VELOCITY_SCALE,
    AltitudeHoldEnv,
    check,
    evaluate,
    hold_controller,
    plot_flights,
    plot_learning_curve,
    run_episode,
)

torch.set_num_threads(1)  # small networks train faster on a single CPU thread
os.makedirs("logs/dqn", exist_ok=True)
os.makedirs("logs/ppo", exist_ok=True)

## Part 1 · Why not a bigger table?

A table needs one entry per state per action. Here is how that grows with the number of inputs, at 10
bins per input:

In [ ]:
for n_inputs in (2, 3, 5, 10, 144):
    print(f"{n_inputs:>3} inputs x 10 bins each -> {3 * 10.0**n_inputs:.3g} table entries")

A table also can't *generalise*: what it learns about one cell tells it nothing about the cell next
door, so every state has to be visited many times.

A neural network solves both problems. It's a single function that maps any observation straight to
three Q-values, one per action. Similar observations give similar outputs, so experience in one state
helps in nearby ones, and adding an input adds a few weights rather than multiplying the table size.

## Part 2 · A Q-network in PyTorch

PyTorch builds networks out of layers:

- `nn.Linear(a, b)` takes `a` inputs and produces `b` outputs, each a weighted sum of the inputs plus a
  bias. The weights and biases are what training adjusts.
- `nn.ReLU()` replaces negative numbers with 0. Without a non-linearity like this between them, a stack
  of linear layers could only ever represent a straight-line (linear) function.

`nn.Sequential(...)` chains layers together. Our network takes the 3 observation numbers, passes them
through two hidden layers of 64 units, and outputs 3 Q-values.

### ✏️ Exercise 4.1 · Build the network

Return `Linear(n_inputs → hidden)`, `ReLU`, `Linear(hidden → hidden)`, `ReLU`, `Linear(hidden → n_actions)`,
chained with `nn.Sequential`. There's no ReLU after the last layer, because Q-values can be negative.

In [ ]:
def build_q_network(n_inputs=3, n_actions=3, hidden=64) -> nn.Sequential:
    # BEGIN SOLUTION
    return nn.Sequential(
        nn.Linear(n_inputs, hidden),
        nn.ReLU(),
        nn.Linear(hidden, hidden),
        nn.ReLU(),
        nn.Linear(hidden, n_actions),
    )
    # END SOLUTION


torch.manual_seed(0)
q_net = build_q_network()
print(q_net)
n_params = sum(p.numel() for p in q_net.parameters())
check(n_params == 4611, f"a 3 -> 64 -> 64 -> 3 network has {n_params:,} weights and biases")

Feed it an observation and it returns one Q-value per action:

In [ ]:
env = AltitudeHoldEnv()
obs, _ = env.reset(seed=0)
with torch.no_grad():
    q_values = q_net(torch.as_tensor(obs))
print("observation:", obs)
print("Q-values:   ", q_values.numpy(), "-> best action:", ACTION_NAMES[int(q_values.argmax())])
check(tuple(q_values.shape) == (3,), "one Q-value per action")

The weights are random, so these Q-values mean nothing yet. To see the whole policy at once, we can
colour every combination of altitude error and velocity by the action the network would pick, like
notebook 3's table but with no bins:

In [ ]:
def plot_action_map(choose_actions, title):
    """Colour each (altitude error, velocity) point by the action `choose_actions` picks there."""
    errors = np.linspace(-1500, 1500, 121)
    velocities = np.linspace(-1.5, 1.5, 61)
    E, V = np.meshgrid(errors, velocities)
    obs_grid = np.stack(
        [E.ravel() / ALTITUDE_SCALE, V.ravel() / VELOCITY_SCALE, np.full(E.size, 0.8)], axis=1
    ).astype(np.float32)
    actions = np.asarray(choose_actions(obs_grid)).reshape(E.shape)
    fig, ax = plt.subplots(figsize=(8, 4))
    cmap = ListedColormap(["tab:blue", "lightgrey", "tab:orange"])
    image = ax.pcolormesh(E, V, actions, cmap=cmap, vmin=-0.5, vmax=2.5, shading="auto")
    ax.axvline(-BAND, color="k", lw=0.5)
    ax.axvline(BAND, color="k", lw=0.5)
    ax.set(xlabel="altitude error (m)", ylabel="vertical velocity (m/s)", title=title)
    colorbar = fig.colorbar(image, ticks=[0, 1, 2])
    colorbar.ax.set_yticklabels(ACTION_NAMES)
    plt.show()


def untrained_actions(obs_grid):
    with torch.no_grad():
        return q_net(torch.as_tensor(obs_grid)).argmax(dim=1).numpy()


plot_action_map(untrained_actions, "Untrained network")

Training will adjust all 4,611 numbers until the outputs match the Q-learning targets. Compare that
with the 324 numbers in notebook 3's table: the network has more parameters, but it never needs bins
and it can generalise.

## Part 3 · From Q-learning to DQN

**DQN** (Deep Q-Network) is the Q-learning from notebook 3 with the table replaced by a network, plus
two tricks that keep training stable:

1. **Replay buffer.** Every transition `(state, action, reward, next state, terminated)` is stored, and
   the network learns from random batches of old and new transitions rather than just the latest step.
   Consecutive steps are very alike, so mixing them up makes learning steadier, and each experience
   gets reused many times.
2. **Target network.** The targets $r + \gamma \max_{a'} Q(s', a')$ are computed with a frozen copy of
   the network, refreshed only every so often. Otherwise the target would move every time the network
   changed, like chasing your own shadow.

And instead of nudging one table cell by α, DQN takes a gradient step that shrinks
$\big(\text{target} - Q(s, a)\big)^2$ across the batch.

Stable-Baselines3 has DQN built in. Here's how notebook 3's settings map onto its arguments:

| Notebook 3 | SB3 `DQN` argument | Value to use |
|---|---|---|
| the Q-table | `policy_kwargs=dict(net_arch=[64, 64])` | two hidden layers of 64, like yours |
| α | `learning_rate` | `1e-3` |
| γ | `gamma` | `0.9` |
| ε from 1 to 0.05 | `exploration_initial_eps`, `exploration_final_eps`, `exploration_fraction` | `1.0`, `0.05`, over the first `0.5` of training |
| (new) replay buffer | `buffer_size`, `batch_size` | `50_000` transitions, batches of `64` |
| (new) | `learning_starts` | act at random for the first `1_000` steps, to fill the buffer |
| (new) | `train_freq` | one gradient step every `4` environment steps |
| (new) target network | `target_update_interval` | refresh it every `1_000` steps |

### ✏️ Exercise 4.2 · Train DQN

Fill in `make_dqn` using the values in the table. Also pass `tensorboard_log="logs/tensorboard"`,
`seed=seed` and `verbose=0`. The first argument is `"MlpPolicy"` (a plain multi-layer network) and the
second is the environment.

The environment is wrapped in `Monitor`, which records every episode's return to a CSV file so we can
plot a learning curve.

In [ ]:
def make_dqn(env, seed=0) -> DQN:
    # BEGIN SOLUTION
    return DQN(
        "MlpPolicy",
        env,
        policy_kwargs=dict(net_arch=[64, 64]),
        learning_rate=1e-3,
        gamma=0.9,
        exploration_initial_eps=1.0,
        exploration_final_eps=0.05,
        exploration_fraction=0.5,
        buffer_size=50_000,
        batch_size=64,
        learning_starts=1_000,
        train_freq=4,
        target_update_interval=1_000,
        tensorboard_log="logs/tensorboard",
        seed=seed,
        verbose=0,
    )
    # END SOLUTION


dqn = make_dqn(Monitor(AltitudeHoldEnv(), "logs/dqn/"))
print(dqn.policy.q_net)
check(dqn.gamma == 0.9 and dqn.learning_rate == 1e-3, "gamma and learning rate set")
check(dqn.policy.net_arch == [64, 64], "two hidden layers of 64")
check(dqn.exploration_fraction == 0.5 and dqn.target_update_interval == 1_000, "exploration and target network set")

Now train for 30,000 steps, which is 750 flights. It takes about 40 seconds.

**TensorBoard (optional).** While it trains, open a terminal in the folder this notebook is in and run
`tensorboard --logdir logs/tensorboard`, then open http://localhost:6006. Look at
`rollout/ep_rew_mean` (the average return), `rollout/exploration_rate` (ε) and `train/loss`. The main
project uses the same tool to watch its training runs.

In [ ]:
%%time
dqn.learn(total_timesteps=30_000, tb_log_name="dqn");

In [ ]:
baselines = {"do nothing": evaluate(lambda obs: STAY), "hold controller": evaluate(hold_controller)}

ax = plot_learning_curve(load_results("logs/dqn")["r"], label="DQN")
ax.axhline(baselines["hold controller"]["total_reward"], color="k", ls="--", label="hold controller")
ax.axhline(baselines["do nothing"]["total_reward"], color="grey", ls=":", label="do nothing")
ax.legend()
plt.show()

To use a trained SB3 model as a policy, call `model.predict(obs, deterministic=True)`. `deterministic`
turns exploration off, just as we did for evaluation in notebook 3.

In [ ]:
def dqn_policy(obs):
    action, _ = dqn.predict(obs, deterministic=True)
    return int(action)


results = dict(baselines, **{"DQN": evaluate(dqn_policy)})
display(pd.DataFrame(results).T.round(2))
check(results["DQN"]["in_band"] > 0.85, "on unseen flights DQN spends over 85% of the time in the band")

flights = [run_episode(env, dqn_policy, seed) for seed in EVAL_SEEDS[4:7]]
plot_flights(flights, labels=["flight 1", "flight 2", "flight 3"], title="DQN")
plt.show()
plot_action_map(lambda obs_grid: dqn.predict(obs_grid, deterministic=True)[0], "Trained DQN")

The action map shows the same pattern as notebook 3's table (drop when low and not rising, vent when
high and not sinking), but with smooth boundaries and no gaps. The network fills in states it rarely
saw from the nearby ones it did.

## Part 4 · Swap the algorithm

Every SB3 algorithm has the same interface, so trying a different one is close to a one-line change.
**PPO** (Proximal Policy Optimisation) is a different family of method. Instead of learning Q-values,
it learns the policy directly: a network that outputs a probability for each action, nudged towards
actions that turned out better than expected.

### ✏️ Exercise 4.3 · PPO

Fill in `make_ppo`: `PPO("MlpPolicy", env, ...)` with `gamma=0.9`, `tensorboard_log="logs/tensorboard"`,
`seed=seed` and `verbose=0`. Leave everything else at SB3's defaults. Then train it on the same 30,000
steps (about a minute).

In [ ]:
def make_ppo(env, seed=0) -> PPO:
    # BEGIN SOLUTION
    return PPO("MlpPolicy", env, gamma=0.9, tensorboard_log="logs/tensorboard", seed=seed, verbose=0)
    # END SOLUTION


ppo = make_ppo(Monitor(AltitudeHoldEnv(), "logs/ppo/"))
check(isinstance(ppo, PPO) and ppo.gamma == 0.9, "PPO created with gamma = 0.9")

In [ ]:
%%time
ppo.learn(total_timesteps=30_000, tb_log_name="ppo");

In [ ]:
def ppo_policy(obs):
    action, _ = ppo.predict(obs, deterministic=True)
    return int(action)


results["PPO"] = evaluate(ppo_policy)
display(pd.DataFrame(results).T.round(2))

ax = plot_learning_curve(load_results("logs/dqn")["r"], label="DQN")
plot_learning_curve(load_results("logs/ppo")["r"], label="PPO", ax=ax)
ax.axhline(baselines["hold controller"]["total_reward"], color="k", ls="--", label="hold controller")
ax.legend()
plt.show()

With the same 30,000 steps PPO gets close to DQN, but its learning curve climbs more slowly, and it
usually ends up a little behind on the evaluation flights (compare the `in_band` column). That's
because the two learn in different ways:

- PPO is **on-policy**: it learns only from experience collected by its current policy, and throws
  that experience away after each update.
- DQN is **off-policy**: its replay buffer lets it reuse every transition many times.

On this small task the gap is modest. On harder ones, how efficiently an algorithm uses experience can
decide whether training takes an hour or a week. Even so, PPO is often the better choice when
experience is cheap (a fast simulator) or the actions are continuous, like a steering angle. Picking an
algorithm is a trade-off, and SB3 makes it cheap to try several.

🚀 **Stretch:** give PPO 100,000 steps (about 3–4 minutes). Does it close the gap?

## Part 5 · From the toy to Stratus

Everything in these notebooks carries over to the main project. Here's what changes:

| | This toy | Stratus |
|---|---|---|
| Task | hold one altitude | stay near a target on the ground by choosing altitudes whose winds blow the right way |
| Dimensions | altitude only | 1D, 2D or 3D: `Balloon3DEnv(dim=...)` |
| Observation | 3 numbers | 144: the wind at 41 levels around the balloon (speed, direction relative to the target, uncertainty), plus 21 numbers such as altitude, distance and heading to the target, and resources left |
| Actions | vent, stay, drop | the same three |
| Decisions | every 3 minutes, 40 per flight | every minute, 720 per 12-hour flight |
| Reward | 1 in the band, decaying outside, −0.1 per action | 1 inside the station-keeping radius, decaying outside, multiplied by a resource-cost factor |
| Algorithm | DQN | QR-DQN from sb3-contrib: a DQN that predicts a spread of possible returns rather than one number, which pays off once the weather is uncertain |
| Network | 2 layers of 64 | 4 layers of 512 |
| Training | 30,000 steps, under a minute on a laptop | 15 million steps, hours on HPC |
| Bar to beat | your hold controller | hand-written baselines (`greedy_wind` in 3D, `bang_bang` in 1D) on held-out scenarios |

Where to look next:

- `src/environments/envs/balloon_3d_env.py`: the real environment, with the same `reset` and `step` you wrote
- `src/environments/core/balloon.py`: the balloon physics you've been flying
- `src/agents/baselines.py`: hand-written controllers, like yours, that agents must beat
- `src/agents/qrdqn.py`: the training set-up. Compare its settings with your `make_dqn`.
- `notes/development_roadmap.md`: where the project is heading, and why

Try `python main.py --benchmark --dim 1` from the repo root. It scores the project's baselines on the
held-out 1D scenarios and takes a couple of minutes.

🚀 **Stretch**
- Retrain DQN with a reward that has no action cost. How many actions does it use per flight now?
- **Steer with the wind:** copy `AltitudeHoldEnv` and add a horizontal position, plus a wind that
  depends on altitude (say, blowing east below 20 km and west above it). Reward staying near position
  0. That's Stratus's real problem in miniature.
- Write DQN yourself in PyTorch, using your `build_q_network`: a replay buffer, a target network, the
  loss and an optimiser.

## Summary

- A neural network replaces the Q-table: no bins, and it generalises between similar states.
- DQN is Q-learning with a network, a replay buffer and a target network.
- SB3 gives you tested algorithms behind one interface, so swapping DQN for PPO is one line, but the
  algorithms differ in how efficiently they use experience.
- The main project is the same loop at a larger scale.

**What next?** Talk to the team about a first issue to pick up. You now know enough to read and change
the real environment and training code.

**Git checkpoint:** `git add onboarding/04_neural_networks_and_sb3.ipynb`, commit and push.